[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

*Attention & Transformers*
Implement **RoPE**: rotate every adjacent pair of head channels by an angle
proportional to the token's position, for both `q` and `k`.

Split the head dimension into $D/2$ pairs. Pair $j$ at position $t$ rotates by

$$\theta_{t,j} = \frac{t}{\text{base}^{\,2j/D}}, \qquad j = 0, \dots, \tfrac{D}{2}-1$$

$$\begin{pmatrix} x'_{2j} \\ x'_{2j+1} \end{pmatrix} =
\begin{pmatrix} \cos\theta_{t,j} & -\sin\theta_{t,j} \\
                \sin\theta_{t,j} & \phantom{-}\cos\theta_{t,j} \end{pmatrix}
\begin{pmatrix} x_{2j} \\ x_{2j+1} \end{pmatrix}$$

### Rules
- Signature: `apply_rope(q, k, base=10000.0) -> (q_rot, k_rot)`
- Input shape `(..., T, D)`: positions run along axis `-2`, channels along axis `-1`,
  `D` even. It must work unchanged for `(B, T, D)` **and** `(B, H, T, Dh)` — build
  `cos`/`sin` as `(T, D/2)` and let broadcasting handle the leading axes
- Positions are `0, 1, ..., T-1`; `q` and `k` get the **same** table
- Use the **interleaved** pairing `(2j, 2j+1)`, not the split-halves variant
- No learned parameters, no lookup table, no `nnx` layers — this is pure math
- Must be jittable and differentiable

### The property that makes it work
Rotation matrices compose: $R_m^\top R_n = R_{n-m}$. So the attention logit

$$\langle R_m q,\; R_n k \rangle = \langle q,\; R_{n-m}\, k \rangle$$

depends **only on the offset** $n - m$, never on where the pair sits in the
window. You inject absolute positions into `q` and `k` and get relative
positions out of the dot product for free — no $O(T^2)$ bias matrix, no extra
parameters, nothing added to `v`.

Two corollaries worth saying out loud in an interview. Each $R$ is orthogonal,
so `‖q_rot‖ = ‖q‖` — RoPE cannot inflate or shrink logits the way an *additive*
position embedding can. And $\theta$ is a closed-form function of $t$, not a row
of a learned table, so position 100,000 is perfectly well defined even if
training stopped at 4,096.

### Why that is *not* free length extrapolation
The usual claim is "RoPE extrapolates." It half does. Any position is
*representable*, but the model has only ever been trained on offsets inside its
window: the low-frequency channels ($j$ near $D/2$, where $\theta \approx
t/10000$) complete only a small fraction of a turn across the whole training
context, so at $10\times$ the length they land at angles the attention heads
have never seen and quality collapses. That gap is the entire reason position
interpolation, NTK-aware base scaling and YaRN exist — they rescale $\theta$ so
inference-time offsets fall back inside the trained range.

### The convention trap
The RoPE paper pairs channels `(0,1), (2,3), ...` — what you are implementing.
Most HuggingFace code instead uses `rotate_half`, pairing `(j, j + D/2)`. Both
give the same relative-offset property; they differ by a fixed permutation of
the head dimension, which a learned `W_q`/`W_k` can absorb. So either is fine to
*train* with, and mixing them is silent corruption at *load* time — the weights
still fit, the loss just quietly goes to garbage.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def apply_rope(q, k, base=10000.0):
    """Apply rotary position embeddings to queries and keys.

    Args:
        q, k: (..., T, D) arrays. Positions along axis -2, channels along -1, D even.
        base: geometric base for the inverse-frequency table.

    Returns:
        (q_rot, k_rot), each the same shape as its input.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

# The frequency table for D=8, base=10000 is exactly [1, 0.1, 0.01, 0.001]:
# pair 0 spins a radian per token, pair 3 barely moves across a whole document.
print("inv_freq:", 1.0 / (10000.0 ** (jnp.arange(4) * 2.0 / 8)))

# Put the SAME vector at every position, then look at the score matrix.
a = jax.random.normal(jax.random.key(0), (16,))
b = jax.random.normal(jax.random.key(1), (16,))
q = jnp.broadcast_to(a, (1, 6, 16))
k = jnp.broadcast_to(b, (1, 6, 16))
qr, kr = apply_rope(q, k)

scores = jnp.einsum("btd,bsd->bts", qr, kr)[0]
print(jnp.round(scores, 3))
print("-> constant along every diagonal: the logit is a function of (n - m) alone")
print("norms preserved:", jnp.allclose(jnp.linalg.norm(q, axis=-1),
                                       jnp.linalg.norm(qr, axis=-1), atol=1e-4))

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("rope")

# hint("rope")      # stuck? nudge without the answer
# solution("rope")  # spoiler: the reference implementation
# status()          # your dashboard across all problems